# Module 5 • Neural Networks for Natural Language Processing

# Lesson 27 • Recurrent Neural Networks for Sequence Modeling

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 150–190 minutes

---

## Scope

This lesson introduces recurrent neural networks as sequence models. It covers
hidden states, sequence unrolling, many-to-one and many-to-many architectures,
Backpropagation Through Time, vanishing and exploding gradients, gradient clipping,
padding masks, bidirectionality, and a complete NumPy RNN text classifier.

No external dataset or model download is required.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain why sequence models differ from pooled feed-forward models;
- describe recurrent hidden states and parameter sharing;
- implement one recurrent step and a full sequence forward pass;
- distinguish many-to-one and many-to-many architectures;
- explain Backpropagation Through Time;
- identify vanishing and exploding gradients;
- apply global-norm gradient clipping;
- use padding masks correctly;
- compare final-state and mean-state sequence representations;
- explain bidirectional recurrence;
- implement, train, and evaluate a NumPy RNN classifier;
- discuss recurrent-model limitations and Arabic sequence modeling.

## Table of Contents

1. Why Sequence Models?
2. Feed-Forward Versus Recurrent Processing
3. Hidden-State Recurrence
4. One RNN Step
5. Unrolling Through Time
6. Shape Reasoning
7. Many-to-One and Many-to-Many Models
8. Order Sensitivity
9. Parameter Sharing
10. Backpropagation Through Time
11. Vanishing and Exploding Gradients
12. Gradient Clipping
13. Truncated BPTT
14. Padding and Masks
15. Sequence Pooling
16. Bidirectional RNNs
17. Classification Dataset
18. Vocabulary and Encoding
19. Parameter Initialization
20. Forward Pass
21. Backward Pass
22. Training Loop
23. Learning Curves
24. Evaluation
25. Error Analysis
26. Hidden-State Inspection
27. Regularization
28. Computational Cost
29. Common Failure Modes
30. Arabic and Multilingual Considerations
31. Reproducibility
32. Knowledge Check
33. Exercises
34. Summary and Next Lesson

# 1. Why Sequence Models?

A pooled feed-forward classifier compresses token vectors without modeling their
order explicitly. An RNN reads tokens sequentially and updates a hidden state that
summarizes the prefix processed so far.

In [ ]:
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

pd.DataFrame(
    [
        ('Mean-pooled feed-forward model', 'order ignored', 'document classification'),
        ('Recurrent neural network', 'order processed sequentially', 'classification and tagging'),
    ],
    columns=['Model', 'Sequence handling', 'Typical uses'],
)

# 2. Feed-Forward Versus Recurrent Processing

```text
Feed-forward: token vectors → pooling → classifier

Recurrent:    x1 → h1 → h2 → h3 → ... → hT
                   ↑    ↑    ↑           ↑
                  x2   x3   x4          xT
```

The same recurrent parameters are reused at every time step.

# 3. Hidden-State Recurrence

A vanilla RNN update is:

$$
h_t = 	anh(x_tW_x + h_{t-1}W_h + b_h)
$$

The hidden state is a learned numerical summary rather than an explicitly
interpretable memory.

In [ ]:
INPUT_DIMENSION = 5
HIDDEN_DIMENSION = 4
rng = np.random.default_rng(42)

x_t = rng.normal(size=(1, INPUT_DIMENSION))
h_previous = np.zeros((1, HIDDEN_DIMENSION))
W_x = rng.normal(0.0, 0.2, size=(INPUT_DIMENSION, HIDDEN_DIMENSION))
W_h = rng.normal(0.0, 0.2, size=(HIDDEN_DIMENSION, HIDDEN_DIMENSION))
b_h = np.zeros(HIDDEN_DIMENSION)

h_t = np.tanh(x_t @ W_x + h_previous @ W_h + b_h)
print('Hidden-state shape:', h_t.shape)

# 4. One RNN Step

In [ ]:
def rnn_step(x_t, h_previous, W_x, W_h, b_h):
    return np.tanh(x_t @ W_x + h_previous @ W_h + b_h)

rnn_step(x_t, h_previous, W_x, W_h, b_h)

# 5. Unrolling Through Time

Conceptually, one recurrent cell is copied across sequence positions, while all
copies share the same parameters.

In [ ]:
def rnn_forward_sequence(inputs, W_x, W_h, b_h, initial_hidden=None):
    batch_size, sequence_length, _ = inputs.shape
    hidden = (
        np.zeros((batch_size, W_h.shape[0]))
        if initial_hidden is None
        else initial_hidden.copy()
    )
    states = []

    for time_step in range(sequence_length):
        hidden = rnn_step(inputs[:, time_step, :], hidden, W_x, W_h, b_h)
        states.append(hidden.copy())

    return np.stack(states, axis=1)

sample_inputs = rng.normal(size=(2, 6, INPUT_DIMENSION))
sample_states = rnn_forward_sequence(sample_inputs, W_x, W_h, b_h)
print(sample_states.shape)

# 6. Shape Reasoning

Common shapes are:

```text
token IDs:      (B, T)
embeddings:     (B, T, E)
hidden states:  (B, T, H)
final state:    (B, H)
class logits:   (B, C)
```

In [ ]:
pd.DataFrame(
    [
        ('Token IDs', '(B, T)'),
        ('Embeddings', '(B, T, E)'),
        ('Hidden states', '(B, T, H)'),
        ('Final state', '(B, H)'),
        ('Class logits', '(B, C)'),
    ],
    columns=['Tensor', 'Shape'],
)

# 7. Many-to-One and Many-to-Many Models

- **Many-to-one:** one output after reading a sequence, such as sentiment or intent.
- **Many-to-many aligned:** one output per input token, such as NER.
- **Sequence-to-sequence:** an input sequence produces another sequence, such as translation.

In [ ]:
pd.DataFrame(
    [
        ('Many-to-one', 'one output', 'text classification'),
        ('Many-to-many aligned', 'one output per token', 'NER'),
        ('Sequence-to-sequence', 'output sequence', 'machine translation'),
    ],
    columns=['Architecture', 'Output pattern', 'Example'],
)

# 8. Order Sensitivity

Unlike mean pooling, an RNN normally produces different states for different token
orders.

In [ ]:
tiny_vectors = {
    'dog': np.array([1.0, 0.0, 0.0]),
    'cat': np.array([0.0, 1.0, 0.0]),
    'chased': np.array([0.0, 0.0, 1.0]),
}
tiny_Wx = rng.normal(0.0, 0.3, size=(3, 4))
tiny_Wh = rng.normal(0.0, 0.3, size=(4, 4))
tiny_b = np.zeros(4)

def word_sequence(words):
    return np.asarray([tiny_vectors[word] for word in words])[None, :, :]

first = rnn_forward_sequence(word_sequence(['dog', 'chased', 'cat']), tiny_Wx, tiny_Wh, tiny_b)[:, -1]
second = rnn_forward_sequence(word_sequence(['cat', 'chased', 'dog']), tiny_Wx, tiny_Wh, tiny_b)[:, -1]
print('Final-state distance:', np.linalg.norm(first - second))

# 9. Parameter Sharing

The recurrent transition matrices are reused at every position. This supports
variable-length sequences and limits parameter count, but requires one transition
function to model all dependency lengths.

# 10. Backpropagation Through Time

Backpropagation Through Time unrolls recurrence and applies the chain rule from later
states to earlier states. Repeated multiplication by recurrent Jacobians is the main
reason gradients can vanish or explode.

# 11. Vanishing and Exploding Gradients

In [ ]:
steps = np.arange(1, 21)
vanishing = 0.6 ** steps

plt.figure(figsize=(8, 5))
plt.plot(steps, vanishing)
plt.title('Illustration of Vanishing Gradient Magnitude')
plt.xlabel('Repeated multiplication count')
plt.ylabel('Magnitude')
plt.tight_layout()
plt.show()

In [ ]:
exploding = 1.35 ** steps

plt.figure(figsize=(8, 5))
plt.plot(steps, exploding)
plt.title('Illustration of Exploding Gradient Magnitude')
plt.xlabel('Repeated multiplication count')
plt.ylabel('Magnitude')
plt.tight_layout()
plt.show()

Vanishing gradients weaken long-range learning. Exploding gradients can destabilize
loss values and produce non-finite parameters.

# 12. Gradient Clipping

Global-norm clipping rescales all gradients when their combined norm exceeds a
threshold.

In [ ]:
def clip_gradients(gradients, max_norm):
    total_norm = math.sqrt(
        sum(float(np.sum(gradient ** 2)) for gradient in gradients.values())
    )

    if total_norm > max_norm and total_norm > 0:
        scale = max_norm / total_norm
        gradients = {name: gradient * scale for name, gradient in gradients.items()}

    return gradients, total_norm

demo_gradients = {'W': np.full((3, 3), 10.0), 'b': np.full(3, 5.0)}
clipped, original_norm = clip_gradients(demo_gradients, 1.0)
clipped_norm = math.sqrt(sum(float(np.sum(value ** 2)) for value in clipped.values()))
print('Original norm:', original_norm)
print('Clipped norm:', clipped_norm)

Clipping controls update magnitude but does not solve vanishing gradients.

# 13. Truncated BPTT

Truncated BPTT limits backward propagation to a fixed number of time steps. It lowers
memory and computation, but dependencies beyond the truncation window receive no
direct gradient.

# 14. Padding and Masks

Padded positions should preserve the previous hidden state rather than creating new
state transitions.

In [ ]:
def masked_rnn_forward(inputs, masks, W_x, W_h, b_h):
    batch_size, sequence_length, _ = inputs.shape
    hidden = np.zeros((batch_size, W_h.shape[0]))
    states = []

    for time_step in range(sequence_length):
        candidate = np.tanh(
            inputs[:, time_step, :] @ W_x + hidden @ W_h + b_h
        )
        mask_t = masks[:, time_step][:, None]
        hidden = mask_t * candidate + (1.0 - mask_t) * hidden
        states.append(hidden.copy())

    return np.stack(states, axis=1)

# 15. Sequence Pooling

Many-to-one models commonly use the final valid hidden state. Mean-state pooling uses
all valid hidden states.

In [ ]:
def final_valid_state(states, masks):
    lengths = np.maximum(masks.sum(axis=1).astype(int), 1)
    return states[np.arange(len(states)), lengths - 1]


def mean_valid_state(states, masks):
    expanded = masks[:, :, None]
    summed = np.sum(states * expanded, axis=1)
    counts = np.maximum(expanded.sum(axis=1), 1.0)
    return summed / counts

# 16. Bidirectional RNNs

A bidirectional RNN combines left-to-right and right-to-left processing. It uses both
past and future context, which helps tagging and offline classification but is not
causal.

In [ ]:
pd.DataFrame(
    [
        ('Forward RNN', 'past context', 'causal processing'),
        ('Backward RNN', 'future context', 'offline processing'),
        ('Bidirectional RNN', 'past and future', 'tagging and classification'),
    ],
    columns=['Direction', 'Context used', 'Typical use'],
)

# 17. Classification Dataset

The dataset contains four balanced intent domains: health, finance, technology, and
travel.

In [ ]:
records = [
    ('doctor treats patient in hospital', 'health'),
    ('nurse gives medicine to patient', 'health'),
    ('patient visits clinic for treatment', 'health'),
    ('hospital schedules medical diagnosis', 'health'),
    ('exercise improves health recovery', 'health'),
    ('nutrition supports patient treatment', 'health'),
    ('doctor reviews the diagnosis', 'health'),
    ('clinic provides medical service', 'health'),
    ('nurse helps patient today', 'health'),
    ('medicine reduces the problem', 'health'),
    ('hospital needs more doctors', 'health'),
    ('patient requests treatment information', 'health'),
    ('medical care supports recovery', 'health'),
    ('doctor works with nurse', 'health'),
    ('clinic checks patient health', 'health'),
    ('treatment starts after diagnosis', 'health'),

    ('bank approves customer loan', 'finance'),
    ('invoice contains payment charge', 'finance'),
    ('customer requests card refund', 'finance'),
    ('billing account has problem', 'finance'),
    ('loan interest increased today', 'finance'),
    ('bank transfers money safely', 'finance'),
    ('payment failed on card', 'finance'),
    ('refund request is pending', 'finance'),
    ('invoice price is incorrect', 'finance'),
    ('customer updates bank account', 'finance'),
    ('billing service changed charge', 'finance'),
    ('loan payment needs approval', 'finance'),
    ('bank reviews the account', 'finance'),
    ('card charge needs refund', 'finance'),
    ('payment invoice arrived today', 'finance'),
    ('interest affects loan price', 'finance'),

    ('software update caused error', 'technology'),
    ('application cannot reach server', 'technology'),
    ('network upload failed today', 'technology'),
    ('computer needs system update', 'technology'),
    ('device cannot install software', 'technology'),
    ('server lost important data', 'technology'),
    ('application displays network error', 'technology'),
    ('computer connects to server', 'technology'),
    ('upload request failed again', 'technology'),
    ('system update needs help', 'technology'),
    ('device reports software problem', 'technology'),
    ('network service is unavailable', 'technology'),
    ('server processes application data', 'technology'),
    ('software installation failed', 'technology'),
    ('computer network needs update', 'technology'),
    ('device connects to system', 'technology'),

    ('flight arrives at airport', 'travel'),
    ('tourist books hotel reservation', 'travel'),
    ('airport lost passenger luggage', 'travel'),
    ('travel ticket changed today', 'travel'),
    ('flight delay affects journey', 'travel'),
    ('hotel reservation needs update', 'travel'),
    ('tourist visits city museum', 'travel'),
    ('beach trip starts tomorrow', 'travel'),
    ('airport changes flight gate', 'travel'),
    ('passenger requests travel information', 'travel'),
    ('journey includes hotel stay', 'travel'),
    ('ticket service reports delay', 'travel'),
    ('tourist carries luggage', 'travel'),
    ('airport checks passenger ticket', 'travel'),
    ('hotel welcomes the tourist', 'travel'),
    ('flight ticket needs approval', 'travel'),
]

dataset = pd.DataFrame(records, columns=['text', 'label'])
dataset['label'].value_counts()

# 18. Vocabulary and Encoding

In [ ]:
TOKEN_PATTERN = re.compile(r"\w+(?:[-']\w+)*", flags=re.UNICODE)

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())

X_train_full, X_test, y_train_full, y_test = train_test_split(
    dataset['text'], dataset['label'], test_size=0.25, random_state=42,
    stratify=dataset['label']
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42,
    stratify=y_train_full
)

training_counts = Counter(token for text in X_train for token in tokenize(text))
vocabulary = ['<PAD>', '<UNK>'] + sorted(training_counts)
word_to_index = {word: index for index, word in enumerate(vocabulary)}
PAD_ID = word_to_index['<PAD>']
UNK_ID = word_to_index['<UNK>']

print('Vocabulary size:', len(vocabulary))
print('Train:', len(X_train), 'Validation:', len(X_validation), 'Test:', len(X_test))

In [ ]:
MAX_LENGTH = 8

def encode_text(text, max_length=MAX_LENGTH):
    token_ids = [word_to_index.get(token, UNK_ID) for token in tokenize(text)][:max_length]
    mask = [1.0] * len(token_ids)
    while len(token_ids) < max_length:
        token_ids.append(PAD_ID)
        mask.append(0.0)
    return np.asarray(token_ids, dtype=int), np.asarray(mask, dtype=float)


def encode_collection(texts):
    encoded = [encode_text(text) for text in texts]
    return np.vstack([item[0] for item in encoded]), np.vstack([item[1] for item in encoded])

train_ids, train_masks = encode_collection(X_train)
validation_ids, validation_masks = encode_collection(X_validation)
test_ids, test_masks = encode_collection(X_test)

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(y_train)
validation_labels = label_encoder.transform(y_validation)
test_labels = label_encoder.transform(y_test)

print(train_ids.shape, train_masks.shape)

The vocabulary is constructed only from the training split. Unknown validation and
test tokens map to `<UNK>`.

# 19. Parameter Initialization

In [ ]:
def initialize_rnn_classifier(vocabulary_size, embedding_dim, hidden_dim, class_count, seed=42):
    generator = np.random.default_rng(seed)
    embeddings = generator.normal(0.0, 0.1, size=(vocabulary_size, embedding_dim))
    embeddings[PAD_ID] = 0.0

    return {
        'embeddings': embeddings,
        'W_x': generator.normal(0.0, math.sqrt(1.0 / embedding_dim), size=(embedding_dim, hidden_dim)),
        'W_h': generator.normal(0.0, math.sqrt(1.0 / hidden_dim), size=(hidden_dim, hidden_dim)),
        'b_h': np.zeros(hidden_dim),
        'W_o': generator.normal(0.0, math.sqrt(1.0 / hidden_dim), size=(hidden_dim, class_count)),
        'b_o': np.zeros(class_count),
    }

model = initialize_rnn_classifier(
    len(vocabulary), embedding_dim=18, hidden_dim=16,
    class_count=len(label_encoder.classes_)
)
{key: value.shape for key, value in model.items()}

# 20. Forward Pass

In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / np.sum(exponentials, axis=1, keepdims=True)


def rnn_classifier_forward(token_ids, masks, parameters):
    embedded = parameters['embeddings'][token_ids]
    batch_size, sequence_length, _ = embedded.shape
    hidden = np.zeros((batch_size, parameters['W_h'].shape[0]))

    states = []
    candidates = []

    for time_step in range(sequence_length):
        candidate = np.tanh(
            embedded[:, time_step, :] @ parameters['W_x']
            + hidden @ parameters['W_h']
            + parameters['b_h']
        )
        mask_t = masks[:, time_step][:, None]
        hidden = mask_t * candidate + (1.0 - mask_t) * hidden
        candidates.append(candidate.copy())
        states.append(hidden.copy())

    states = np.stack(states, axis=1)
    candidates = np.stack(candidates, axis=1)
    representation = final_valid_state(states, masks)
    logits = representation @ parameters['W_o'] + parameters['b_o']
    probabilities = softmax(logits)

    cache = {
        'token_ids': token_ids,
        'masks': masks,
        'embedded': embedded,
        'states': states,
        'candidates': candidates,
        'representation': representation,
        'probabilities': probabilities,
    }
    return probabilities, cache


def cross_entropy_loss(probabilities, labels):
    selected = probabilities[np.arange(len(labels)), labels]
    return float(-np.mean(np.log(np.clip(selected, 1e-12, 1.0))))

probabilities, forward_cache = rnn_classifier_forward(train_ids[:4], train_masks[:4], model)
print(probabilities.shape)
print('Loss:', cross_entropy_loss(probabilities, train_labels[:4]))

# 21. Backward Pass

The final-state gradient is injected at the last valid position of each sequence and
then propagated backward through recurrent transitions.

In [ ]:
def one_hot(labels, class_count):
    encoded = np.zeros((len(labels), class_count))
    encoded[np.arange(len(labels)), labels] = 1.0
    return encoded


def rnn_classifier_backward(labels, cache, parameters, l2_strength=0.0):
    batch_size, sequence_length = cache['token_ids'].shape
    targets = one_hot(labels, parameters['b_o'].shape[0])
    d_logits = (cache['probabilities'] - targets) / batch_size

    gradients = {
        'embeddings': np.zeros_like(parameters['embeddings']),
        'W_x': np.zeros_like(parameters['W_x']),
        'W_h': np.zeros_like(parameters['W_h']),
        'b_h': np.zeros_like(parameters['b_h']),
        'W_o': cache['representation'].T @ d_logits + l2_strength * parameters['W_o'],
        'b_o': d_logits.sum(axis=0),
    }

    d_representation = d_logits @ parameters['W_o'].T
    lengths = np.maximum(cache['masks'].sum(axis=1).astype(int), 1)
    d_hidden_next = np.zeros_like(d_representation)

    for time_step in reversed(range(sequence_length)):
        mask_t = cache['masks'][:, time_step][:, None]
        candidate = cache['candidates'][:, time_step, :]
        previous_state = (
            np.zeros_like(candidate)
            if time_step == 0
            else cache['states'][:, time_step - 1, :]
        )

        is_final = ((lengths - 1) == time_step)[:, None]
        d_hidden = d_hidden_next + is_final * d_representation

        d_candidate = d_hidden * mask_t
        d_previous_skip = d_hidden * (1.0 - mask_t)
        d_preactivation = d_candidate * (1.0 - candidate ** 2)

        gradients['W_x'] += cache['embedded'][:, time_step, :].T @ d_preactivation
        gradients['W_h'] += previous_state.T @ d_preactivation
        gradients['b_h'] += d_preactivation.sum(axis=0)

        d_embedding = d_preactivation @ parameters['W_x'].T
        for row in range(batch_size):
            token_id = cache['token_ids'][row, time_step]
            if token_id != PAD_ID and cache['masks'][row, time_step] > 0:
                gradients['embeddings'][token_id] += d_embedding[row]

        d_hidden_next = d_preactivation @ parameters['W_h'].T + d_previous_skip

    gradients['W_x'] += l2_strength * parameters['W_x']
    gradients['W_h'] += l2_strength * parameters['W_h']
    gradients['embeddings'][PAD_ID] = 0.0
    return gradients

gradients = rnn_classifier_backward(train_labels[:4], forward_cache, model)
{key: value.shape for key, value in gradients.items()}

# 22. Training Loop

In [ ]:
def batch_indices(example_count, batch_size, generator):
    indices = np.arange(example_count)
    generator.shuffle(indices)
    for start in range(0, example_count, batch_size):
        yield indices[start:start + batch_size]


def evaluate_rnn(token_ids, masks, labels, parameters):
    probabilities, _ = rnn_classifier_forward(token_ids, masks, parameters)
    predictions = probabilities.argmax(axis=1)
    return {
        'loss': cross_entropy_loss(probabilities, labels),
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro'),
        'predictions': predictions,
        'probabilities': probabilities,
    }

In [ ]:
def train_rnn(
    parameters,
    epochs=180,
    batch_size=8,
    learning_rate=0.035,
    l2_strength=1e-4,
    clip_norm=5.0,
    patience=25,
    seed=42,
):
    generator = np.random.default_rng(seed)
    best_parameters = {key: value.copy() for key, value in parameters.items()}
    best_validation_loss = float('inf')
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        current_rate = learning_rate * (1.0 - 0.75 * epoch / max(epochs - 1, 1))
        gradient_norms = []

        for indices in batch_indices(len(train_ids), batch_size, generator):
            _, cache = rnn_classifier_forward(
                train_ids[indices], train_masks[indices], parameters
            )
            gradients = rnn_classifier_backward(
                train_labels[indices], cache, parameters, l2_strength=l2_strength
            )
            gradients, original_norm = clip_gradients(gradients, clip_norm)
            gradient_norms.append(original_norm)

            for key in parameters:
                parameters[key] -= current_rate * gradients[key]
            parameters['embeddings'][PAD_ID] = 0.0

        train_metrics = evaluate_rnn(train_ids, train_masks, train_labels, parameters)
        validation_metrics = evaluate_rnn(
            validation_ids, validation_masks, validation_labels, parameters
        )

        history.append({
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'validation_loss': validation_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'validation_accuracy': validation_metrics['accuracy'],
            'mean_gradient_norm': float(np.mean(gradient_norms)),
        })

        if validation_metrics['loss'] < best_validation_loss - 1e-5:
            best_validation_loss = validation_metrics['loss']
            best_parameters = {key: value.copy() for key, value in parameters.items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    return best_parameters, pd.DataFrame(history)

trained_model, training_history = train_rnn(model)
print('Epochs completed:', len(training_history))

# 23. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_history['epoch'], training_history['train_loss'], label='Training loss')
plt.plot(training_history['epoch'], training_history['validation_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('RNN Learning Curves')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_history['epoch'], training_history['mean_gradient_norm'])
plt.xlabel('Epoch')
plt.ylabel('Mean pre-clipping gradient norm')
plt.title('RNN Gradient Norms')
plt.tight_layout()
plt.show()

# 24. Evaluation

In [ ]:
test_metrics = evaluate_rnn(test_ids, test_masks, test_labels, trained_model)
print('Test loss:', round(test_metrics['loss'], 3))
print('Test accuracy:', round(test_metrics['accuracy'], 3))
print('Test macro F1:', round(test_metrics['macro_f1'], 3))

In [ ]:
predicted_labels = label_encoder.inverse_transform(test_metrics['predictions'])
print(classification_report(y_test, predicted_labels, zero_division=0))

In [ ]:
class_names = list(label_encoder.classes_)
confusion = confusion_matrix(y_test, predicted_labels, labels=class_names)
pd.DataFrame(
    confusion,
    index=[f'actual_{label}' for label in class_names],
    columns=[f'predicted_{label}' for label in class_names],
)

# 25. Error Analysis

In [ ]:
error_frame = pd.DataFrame({
    'text': X_test.reset_index(drop=True),
    'actual': y_test.reset_index(drop=True),
    'predicted': predicted_labels,
    'confidence': test_metrics['probabilities'].max(axis=1),
})
error_frame['correct'] = error_frame['actual'] == error_frame['predicted']
error_frame.sort_values(['correct', 'confidence'], ascending=[True, True])

Review errors for OOV words, truncation, mixed domains, insufficient context, label
ambiguity, and unstable long-range dependencies.

# 26. Hidden-State Inspection

Hidden-state trajectories can be inspected, but individual dimensions should not be
interpreted as isolated linguistic features.

In [ ]:
_, sample_cache = rnn_classifier_forward(test_ids[:1], test_masks[:1], trained_model)
hidden_norms = np.linalg.norm(sample_cache['states'][0], axis=1)

pd.DataFrame({
    'position': np.arange(len(hidden_norms)),
    'token_id': test_ids[0],
    'mask': test_masks[0],
    'hidden_norm': hidden_norms,
})

In [ ]:
final_representations = final_valid_state(sample_cache['states'], test_masks[:1])
mean_representations = mean_valid_state(sample_cache['states'], test_masks[:1])
print('Final-versus-mean distance:', np.linalg.norm(final_representations - mean_representations))

# 27. Regularization

RNN regularization may include L2 penalties, embedding dropout, hidden-output dropout,
recurrent dropout, early stopping, smaller hidden states, and more training data.

In [ ]:
pd.DataFrame(
    [
        ('L2', 'discourages large weights'),
        ('Embedding dropout', 'drops token features'),
        ('Hidden dropout', 'regularizes sequence representations'),
        ('Early stopping', 'limits overfitting'),
        ('Smaller hidden state', 'reduces model capacity'),
    ],
    columns=['Method', 'Purpose'],
)

# 28. Computational Cost

RNN time steps are sequentially dependent. This limits parallelism across positions
and increases training time for long sequences.

In [ ]:
pd.DataFrame(
    [
        ('Sequence length', 'more recurrent steps', 'greater time and BPTT memory'),
        ('Hidden dimension', 'larger recurrent matrix', 'higher quadratic hidden cost'),
        ('Batch size', 'more sequences per update', 'greater memory use'),
    ],
    columns=['Factor', 'Cause', 'Effect'],
)

# 29. Common Failure Modes

In [ ]:
pd.DataFrame(
    [
        ('Vanishing gradients', 'use LSTM, GRU, attention, or shorter paths'),
        ('Exploding gradients', 'clip gradients'),
        ('Padding contamination', 'use masks'),
        ('Poor long-range memory', 'use gated recurrence'),
        ('Overfitting', 'regularize and stop early'),
        ('Slow sequential computation', 'consider parallel architectures'),
    ],
    columns=['Failure', 'Possible response'],
)

# 30. Arabic and Multilingual Considerations

Arabic sequence modeling must account for attached clitics, rich morphology,
optional diacritics, orthographic variants, MSA and dialects, Arabizi,
code-switching, and tokenization granularity.

In [ ]:
pd.DataFrame(
    [
        ('وَبِالْمَدْرَسَةِ', 'وَ + بِ + الْمَدْرَسَةِ'),
        ('سَيَكْتُبُونَهَا', 'سَ + يَكْتُبُونَ + هَا'),
        ('كِتَابُهُ', 'كِتَابُ + هُ'),
    ],
    columns=['Surface form', 'Illustrative segmentation'],
)

In [ ]:
pd.DataFrame(
    [
        ('Word', 'shorter sequence', 'larger vocabulary'),
        ('Subword', 'moderate sequence', 'better coverage'),
        ('Character', 'long sequence', 'small vocabulary'),
        ('Morphological segment', 'linguistically informed', 'requires an analyzer'),
    ],
    columns=['Unit', 'Sequence effect', 'Trade-off'],
)

For fully vocalized Arabic tasks, diacritics should not be removed unless the
experimental design explicitly tests that transformation.

# 31. Reproducibility and Reporting

Report the dataset, split, tokenizer, vocabulary threshold, maximum sequence length,
embedding and hidden dimensions, recurrence type, pooling strategy, initialization,
optimizer, learning-rate schedule, batch size, gradient clipping, regularization,
early stopping, random seed, and evaluation metrics.

In [ ]:
import platform

pd.Series(
    {
        'dataset_examples': len(dataset),
        'classes': dataset['label'].nunique(),
        'vocabulary_size': len(vocabulary),
        'maximum_length': MAX_LENGTH,
        'embedding_dimension': trained_model['embeddings'].shape[1],
        'hidden_dimension': trained_model['W_h'].shape[0],
        'pooling': 'final valid hidden state',
        'gradient_clipping': 5.0,
        'random_seed': 42,
        'python_version': platform.python_version(),
        'numpy_version': np.__version__,
    },
    name='RNN experiment',
)

# 32. Knowledge Check

1. Why are RNNs sequence models?
2. What does the hidden state represent?
3. Which parameters are shared across time?
4. What does unrolling mean?
5. How do many-to-one and many-to-many models differ?
6. Why is an RNN sensitive to token order?
7. What is Backpropagation Through Time?
8. Why do gradients vanish or explode?
9. What does gradient clipping do?
10. What is truncated BPTT?
11. Why are masks required?
12. How do final-state and mean-state pooling differ?
13. What context does a bidirectional RNN use?
14. Why are RNNs difficult to parallelize across time?
15. Which Arabic properties affect sequence length and vocabulary?

# 33. Exercises

## Exercise 1 — Recurrent Step

Calculate one hidden-state update manually.

## Exercise 2 — Sequence Unrolling

Trace hidden states for a three-token sequence.

## Exercise 3 — Order Sensitivity

Compare representations for several token-order permutations.

## Exercise 4 — Gradient Clipping

Compare training with and without clipping.

## Exercise 5 — Hidden Dimension

Train models with hidden dimensions 8, 16, and 32.

## Exercise 6 — Pooling

Compare final-state and mean-state classifiers.

## Exercise 7 — Bidirectional RNN

Implement separate forward and backward encoders.

## Exercise 8 — Sequence Labeling

Modify the model to produce one label per token.

## Exercise 9 — Arabic Classification

Compare word, subword, and character tokenization.

## Exercise 10 — Long Sequences

Measure the effect of truncation length.

## Challenge Exercises

1. Implement truncated BPTT.
2. Add recurrent dropout.
3. Add momentum or Adam optimization.
4. Implement a bidirectional classifier.
5. Refactor the RNN into reusable Python classes.

# 34. Summary and Next Lesson

In this lesson:

- recurrent networks were introduced as order-sensitive sequence models;
- hidden states summarized prefixes;
- recurrent parameters were shared across time;
- many-to-one and many-to-many architectures were distinguished;
- BPTT propagated gradients through recurrent transitions;
- vanishing and exploding gradients were illustrated;
- global-norm clipping stabilized updates;
- masks prevented padding from changing valid states;
- final-state and mean-state pooling were compared;
- bidirectional processing was introduced;
- a complete NumPy RNN classifier was trained and evaluated;
- Arabic morphology and tokenization were connected to sequence design.

## Next Lesson

**Lesson 28: LSTM and GRU Networks for Long-Range Dependencies** introduces memory
cells, input and forget gates, update and reset gates, and practical gated recurrent
classification.

# References

- Elman, J. L. recurrent-network literature.
- Rumelhart, D. E., Hinton, G. E., & Williams, R. J. backpropagation literature.
- Goodfellow, I., Bengio, Y., & Courville, A. *Deep Learning*.
- Goldberg, Y. *Neural Network Methods for Natural Language Processing*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.